<a href="https://colab.research.google.com/github/Benevalterjr/algo-lecum/blob/main/LECUM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch numpy

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Encoder(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
class Predictor(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )

    def forward(self, z_context):
        return self.net(z_context)

In [5]:
class EnergyModel(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, z_context, z_candidate):
        x = torch.cat([z_context, z_candidate], dim=-1)
        return self.net(x)

In [6]:
def generate_data(num_samples=10000, input_dim=64):
    X = torch.randn(num_samples, input_dim)
    Y = X + 0.1 * torch.randn(num_samples, input_dim)  # "futuro"
    return X, Y

X, Y = generate_data()

In [7]:
encoder = Encoder().to(device)
predictor = Predictor().to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) + list(predictor.parameters()),
    lr=1e-3
)

loss_fn = nn.MSELoss()

X = X.to(device)
Y = Y.to(device)

for epoch in range(10):
    optimizer.zero_grad()

    z_x = encoder(X)
    z_y = encoder(Y)

    z_pred = predictor(z_x)

    loss = loss_fn(z_pred, z_y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch} - JEPA Loss: {loss.item():.4f}")

Epoch 0 - JEPA Loss: 0.0691
Epoch 1 - JEPA Loss: 0.0567
Epoch 2 - JEPA Loss: 0.0484
Epoch 3 - JEPA Loss: 0.0425
Epoch 4 - JEPA Loss: 0.0382
Epoch 5 - JEPA Loss: 0.0348
Epoch 6 - JEPA Loss: 0.0321
Epoch 7 - JEPA Loss: 0.0298
Epoch 8 - JEPA Loss: 0.0277
Epoch 9 - JEPA Loss: 0.0260


In [8]:
energy_model = EnergyModel().to(device)

optimizer_energy = optim.Adam(energy_model.parameters(), lr=1e-3)

for epoch in range(10):
    optimizer_energy.zero_grad()

    z_x = encoder(X)
    z_y = encoder(Y)

    # positivos (corretos)
    pos_energy = energy_model(z_x, z_y)

    # negativos (embaralhados)
    shuffled = z_y[torch.randperm(z_y.size(0))]
    neg_energy = energy_model(z_x, shuffled)

    loss = torch.mean(pos_energy) - torch.mean(neg_energy)

    loss.backward()
    optimizer_energy.step()

    print(f"Epoch {epoch} - Energy Loss: {loss.item():.4f}")

Epoch 0 - Energy Loss: 0.0018
Epoch 1 - Energy Loss: -0.0046
Epoch 2 - Energy Loss: -0.0112
Epoch 3 - Energy Loss: -0.0179
Epoch 4 - Energy Loss: -0.0243
Epoch 5 - Energy Loss: -0.0312
Epoch 6 - Energy Loss: -0.0385
Epoch 7 - Energy Loss: -0.0460
Epoch 8 - Energy Loss: -0.0536
Epoch 9 - Energy Loss: -0.0619


In [12]:
def select_best(context, candidates):
    context = context.clone().detach().float().to(device)
    context = context.unsqueeze(0)

    z_context = encoder(context)

    energies = []

    for c in candidates:
        c = c.clone().detach().float().unsqueeze(0).to(device)
        z_c = encoder(c)

        energy = energy_model(z_context, z_c)
        energies.append(energy.item())

    best_idx = torch.argmin(torch.tensor(energies))
    return candidates[best_idx], energies

In [13]:
context = torch.randn(64)

candidates = [
    context + 0.1 * torch.randn(64),  # plausível
    torch.randn(64),                  # aleatório
    context + 2 * torch.randn(64)     # ruim
]

best, energies = select_best(context, candidates)

print("Energias:", energies)
print("Melhor candidato escolhido:", best[:5])

Energias: [-0.159339040517807, -0.015055900439620018, -0.28132936358451843]
Melhor candidato escolhido: tensor([-2.0183, -2.7987, -3.0138,  2.5546, -1.8374])


### 🧠 Integrating Language: The 'Narrator' Interface
Now we bridge the gap between the JEPA latent space and human language. We'll implement a simple text decoder and a logic-based narrator that uses the Energy Model's output to describe the 'plausibility' of a scenario.

In [14]:
class LatentNarrator(nn.Module):
    def __init__(self, latent_dim=128, vocab_size=50):
        super().__init__()
        # A simple mapping from latent space to 'semantic tags'
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, vocab_size)
        )

    def forward(self, z):
        return self.decoder(z)

def get_narrative_feedback(energy_val):
    """Option C: Logic-based template narrating based on Energy scores"""
    if energy_val < -0.2:
        return "✅ HIGHLY PLAUSIBLE: This scenario aligns perfectly with historical patterns."
    elif energy_val < -0.1:
        return "⚠️ MODERATE: The scenario is consistent but contains some noise."
    else:
        return "❌ UNLIKELY: High energy detected. This state is inconsistent with the model's logic."

# Initialize a mock narrator
narrator = LatentNarrator().to(device)
print("Narrator interface initialized.")

Narrator interface initialized.


In [15]:
# Final Demo: From Latent Energy to Human Language
print(f"--- JEPA Decision Analysis ---\n")

for i, energy in enumerate(energies):
    status = get_narrative_feedback(energy)
    print(f"Candidate {i+1}:")
    print(f"  Energy Score: {energy:.4f}")
    print(f"  Model Verdict: {status}\n")

print(f"Final Decision: Choosing Candidate {torch.argmin(torch.tensor(energies)) + 1}")

--- JEPA Decision Analysis ---

Candidate 1:
  Energy Score: -0.1593
  Model Verdict: ⚠️ MODERATE: The scenario is consistent but contains some noise.

Candidate 2:
  Energy Score: -0.0151
  Model Verdict: ❌ UNLIKELY: High energy detected. This state is inconsistent with the model's logic.

Candidate 3:
  Energy Score: -0.2813
  Model Verdict: ✅ HIGHLY PLAUSIBLE: This scenario aligns perfectly with historical patterns.

Final Decision: Choosing Candidate 3


### 📈 Advanced Metrics: Probabilities & Gap Analysis
To make the system more robust, we'll convert raw energy into probabilities and calculate the 'decision gap' to measure model confidence.

In [16]:
import torch.nn.functional as F

def get_advanced_analysis(energies):
    energy_tensor = torch.tensor(energies)
    # Lower energy is better, so we use negative energy for softmax
    probs = F.softmax(-energy_tensor, dim=0)

    # Calculate the gap between the best and second best
    sorted_energies, _ = torch.sort(energy_tensor)
    gap = abs(sorted_energies[0] - sorted_energies[1])

    confidence_level = "HIGH" if gap > 0.1 else "LOW"
    return probs, gap.item(), confidence_level

probs, gap, confidence = get_advanced_analysis(energies)

print(f"--- Advanced Confidence Report ---")
print(f"Probability Distribution: {probs.tolist()}")
print(f"Decision Gap: {gap:.4f} ({confidence} confidence)")

--- Advanced Confidence Report ---
Probability Distribution: [0.3338468074798584, 0.2889920175075531, 0.3771611452102661]
Decision Gap: 0.1220 (HIGH confidence)


In [18]:
# Final Calibrated Output with realistic confidence thresholds
best_idx = torch.argmax(probs).item()
sorted_probs, _ = torch.sort(probs, descending=True)
prob_gap = (sorted_probs[0] - sorted_probs[1]).item()

print(f"--- Final Model Verdict ---")
print(f"Selected: Candidate {best_idx + 1}")
print(f"Confidence: {probs[best_idx]*100:.1f}%")
print(f"Probability Gap: {prob_gap*100:.1f}%")

if probs[best_idx] < 0.4 or prob_gap < 0.1:
    print("\n⚠️ STATUS: AMBIGUOUS PREFERENCE")
    print("The system sees multiple plausible paths. Candidate " +
          f"{best_idx + 1} leads marginally, but alternatives are nearly as consistent.")
    print("Strategic Recommendation: Analyze the Top 2 candidates as a cluster.")
else:
    print("\n✅ STATUS: CLEAR STRATEGIC ALIGNMENT")
    print("The model shows a strong preference for this path with significant separation from alternatives.")

--- Final Model Verdict ---
Selected: Candidate 3
Confidence: 37.7%
Probability Gap: 4.3%

⚠️ STATUS: AMBIGUOUS PREFERENCE
The system sees multiple plausible paths. Candidate 3 leads marginally, but alternatives are nearly as consistent.
Strategic Recommendation: Analyze the Top 2 candidates as a cluster.


### 🛠️ Strategic Action Layer
Based on the confidence metrics, we now automate the next logical step in the decision pipeline. This ensures the model doesn't just 'output a value' but directs the workflow.

In [19]:
def determine_next_action(prob_val, gap_val):
    if prob_val > 0.6 and gap_val > 0.2:
        return "🚀 EXECUTE: Proceed with high-confidence implementation."
    elif gap_val < 0.05:
        return "🔄 RE-SAMPLE: Generate 10x more candidates via ACO to break the tie."
    elif prob_val < 0.4:
        return "📚 EXPAND CONTEXT: Current latent features are insufficient for a clear path."
    else:
        return "⚖️ REVIEW: Human-in-the-loop validation recommended for top 2 paths."

action = determine_next_action(probs[best_idx].item(), prob_gap)
print(f"--- Automated Strategic Directive ---")
print(f"Recommended Action: {action}")

--- Automated Strategic Directive ---
Recommended Action: 🔄 RE-SAMPLE: Generate 10x more candidates via ACO to break the tie.


## 📈 Real-World Application: Adaptive Market Analysis
We'll now apply our Energy-Based Decision framework to real financial assets. We'll use `yfinance` for data, and implement an automated loop that re-samples candidates if the initial decision is ambiguous.

In [ ]:
!pip install yfinance

In [21]:
import yfinance as yf
import pandas as pd
import numpy as np

def get_market_context(ticker):
    data = yf.download(ticker, period="1mo", interval="1d", progress=False)
    if data.empty: return None

    # Feature Engineering: Returns and Volatility
    returns = data['Close'].pct_change().dropna()
    vol = returns.rolling(window=5).std().dropna()

    # Context: Last 5 days of returns + last 5 days of volatility
    context = np.concatenate([returns.values[-5:], vol.values[-5:]])
    return torch.tensor(context).float().to(device)

def generate_market_candidates(n=5, dim=10):
    # Generate random future 'paths' (hypothetical next-day behaviors)
    return [torch.randn(dim).to(device) for _ in range(n)]

def guided_resample(best_c, n=5):
    # ACO-inspired: Exploration around the current best candidate
    return [best_c + 0.05 * torch.randn_like(best_c) for _ in range(n)]

print("Market Analysis functions ready.")

Market Analysis functions ready.


In [24]:
# Execute Adaptive Pipeline for a specific Ticker
ticker = "PETR4.SA"
market_context = get_market_context(ticker)

if market_context is not None:
    # FIX: Flatten market_context and pad to 64 to match the Encoder's expected input_dim
    market_context_flat = market_context.flatten()
    padded_context = torch.zeros(64).to(device)
    padded_context[:len(market_context_flat)] = market_context_flat

    print(f"--- Analysis for {ticker} ---")

    # 1. Initial Evaluation
    # Candidates must also match the input_dim (64)
    initial_candidates = [torch.randn(64).to(device) for _ in range(5)]

    # Re-use our existing JEPA/Energy infrastructure
    z_context = encoder(padded_context.unsqueeze(0))

    initial_energies = []
    for c in initial_candidates:
        z_c = encoder(c.unsqueeze(0))
        e = energy_model(z_context, z_c)
        initial_energies.append(e.item())

    probs, gap, conf = get_advanced_analysis(initial_energies)
    best_idx = torch.argmax(probs).item()

    print(f"Initial Best Prob: {probs[best_idx]:.2%}")
    print(f"Initial Gap: {gap:.4f} ({conf} confidence)")

    # 2. Adaptive Loop: Re-sample if confidence is low
    if conf == "LOW":
        print("\n🔄 Triggering ACO-Guided Re-sampling...")
        best_c = initial_candidates[best_idx]
        refined_candidates = guided_resample(best_c, n=10)

        refined_energies = []
        for c in refined_candidates:
            z_c = encoder(c.unsqueeze(0))
            e = energy_model(z_context, z_c)
            refined_energies.append(e.item())

        probs_new, gap_new, conf_new = get_advanced_analysis(refined_energies)
        best_idx_new = torch.argmax(probs_new).item()

        print(f"Refined Best Prob: {probs_new[best_idx_new]:.2%}")
        print(f"Refined Gap: {gap_new:.4f} ({conf_new} confidence)")

        final_action = determine_next_action(probs_new[best_idx_new].item(), gap_new)
    else:
        final_action = determine_next_action(probs[best_idx].item(), gap)

    print(f"\nFINAL STRATEGIC DIRECTIVE: {final_action}")

--- Analysis for PETR4.SA ---
Initial Best Prob: 20.24%
Initial Gap: 0.0070 (LOW confidence)

🔄 Triggering ACO-Guided Re-sampling...
Refined Best Prob: 10.03%
Refined Gap: 0.0019 (LOW confidence)

FINAL STRATEGIC DIRECTIVE: 🔄 RE-SAMPLE: Generate 10x more candidates via ACO to break the tie.


/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)


### 🎓 Contrastive Training: Learning Market Dynamics
Currently, our Energy Model is random. We need to train it so that real historical transitions have **low energy** and random transitions have **high energy**. This is the core of Energy-Based Models (EBM).

In [25]:
def train_energy_model(model, ticker_data, epochs=50):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Prepare training pairs (Context -> Real Future)
    returns = ticker_data['Close'].pct_change().dropna().values

    print(f"Training on real patterns...")
    for epoch in range(epochs):
        total_loss = 0
        for i in range(len(returns) - 10):
            optimizer.zero_grad()

            # Context (5 days) and Real Future (Next 5 days)
            ctx = torch.tensor(returns[i:i+5]).float().to(device).flatten()
            real_next = torch.tensor(returns[i+5:i+10]).float().to(device).flatten()

            # Pad to 64 to match encoder
            ctx_p = torch.zeros(64).to(device); ctx_p[:5] = ctx
            real_p = torch.zeros(64).to(device); real_p[:5] = real_next

            # Fake Future (Noise)
            fake_p = torch.randn(64).to(device) * 0.02

            # Latent representations
            z_ctx = encoder(ctx_p.unsqueeze(0))
            z_real = encoder(real_p.unsqueeze(0))
            z_fake = encoder(fake_p.unsqueeze(0))

            # Contrastive Loss: Real should have LOWER energy than Fake
            pos_energy = energy_model(z_ctx, z_real)
            neg_energy = energy_model(z_ctx, z_fake)

            loss = torch.relu(pos_energy - neg_energy + 1.0) # Margin Loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss: {total_loss/len(returns):.4f}")

# Run the training
data_full = yf.download("PETR4.SA", period="1y", progress=False)
train_energy_model(energy_model, data_full)

/tmp/ipykernel_40346/3126148874.py:42: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_full = yf.download("PETR4.SA", period="1y", progress=False)


Training on real patterns...
Epoch 0 | Loss: 0.9578
Epoch 10 | Loss: 0.4841
Epoch 20 | Loss: 0.2334
Epoch 30 | Loss: 0.2162
Epoch 40 | Loss: 0.1128


### 🏁 Multi-Asset Validation: Testing the Trained Brain
We will now run our trained Energy-Based Model across different tickers to see if it provides higher confidence and clearer strategic separation.

In [27]:
tickers = ["PETR4.SA", "VALE3.SA", "ITUB4.SA"]

for t in tickers:
    print(f"\n🔍 Analyzing {t}...")
    ctx_raw = get_market_context(t)

    if ctx_raw is not None:
        # Prepare context
        ctx_flat = ctx_raw.flatten()
        padded_ctx = torch.zeros(64).to(device)
        padded_ctx[:len(ctx_flat)] = ctx_flat

        # Generate candidates
        test_candidates = [torch.randn(64).to(device) * 0.05 for _ in range(5)]

        # Evaluate
        z_ctx = encoder(padded_ctx.unsqueeze(0))
        test_energies = []
        for c in test_candidates:
            z_c = encoder(c.unsqueeze(0))
            test_energies.append(energy_model(z_ctx, z_c).item())

        # Analysis
        t_probs, t_gap, t_conf = get_advanced_analysis(test_energies)
        t_best = torch.argmax(t_probs).item()

        print(f"  Best Scenario Prob: {t_probs[t_best]:.2%}")
        print(f"  Energy Gap: {t_gap:.4f} ({t_conf} confidence)")

        # Final Logic
        action = determine_next_action(t_probs[t_best].item(), t_gap)
        print(f"  STRATEGIC DIRECTIVE: {action}")


🔍 Analyzing PETR4.SA...
  Best Scenario Prob: 49.75%
  Energy Gap: 0.2987 (HIGH confidence)
  STRATEGIC DIRECTIVE: ⚖️ REVIEW: Human-in-the-loop validation recommended for top 2 paths.

🔍 Analyzing VALE3.SA...


/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)
/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)


  Best Scenario Prob: 84.95%
  Energy Gap: 2.3124 (HIGH confidence)
  STRATEGIC DIRECTIVE: 🚀 EXECUTE: Proceed with high-confidence implementation.

🔍 Analyzing ITUB4.SA...


/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)


  Best Scenario Prob: 82.20%
  Energy Gap: 1.8706 (HIGH confidence)
  STRATEGIC DIRECTIVE: 🚀 EXECUTE: Proceed with high-confidence implementation.


### 🚀 Multi-Asset Strategy Scanner
This cell formalizes the strategic layer by ranking assets based on their **Decision Confidence** (Probability Gap) and generating a consolidated report for portfolio allocation.

In [28]:
def run_market_scanner(tickers):
    results = []

    for t in tickers:
        ctx_raw = get_market_context(t)
        if ctx_raw is None: continue

        # Prepare and Evaluate
        ctx_flat = ctx_raw.flatten()
        padded_ctx = torch.zeros(64).to(device)
        padded_ctx[:len(ctx_flat)] = ctx_flat

        # Generate candidates (future paths)
        test_candidates = [torch.randn(64).to(device) * 0.05 for _ in range(5)]

        z_ctx = encoder(padded_ctx.unsqueeze(0))
        energies = []
        for c in test_candidates:
            z_c = encoder(c.unsqueeze(0))
            energies.append(energy_model(z_ctx, z_c).item())

        probs, gap, conf = get_advanced_analysis(energies)
        best_idx = torch.argmax(probs).item()

        results.append({
            'Ticker': t,
            'Confidence_Prob': probs[best_idx].item(),
            'Energy_Gap': gap,
            'Status': conf,
            'Directive': determine_next_action(probs[best_idx].item(), gap)
        })

    # Rank by Energy Gap (Structural Certainty)
    df_report = pd.DataFrame(results).sort_values(by='Energy_Gap', ascending=False)
    return df_report

# Execution
scanner_report = run_market_scanner(["PETR4.SA", "VALE3.SA", "ITUB4.SA", "ABEV3.SA", "BBDC4.SA"])

print("--- 🎯 STRATEGIC SCANNER REPORT ---")
display(scanner_report)

# Automated Filtering for High-Conviction
high_conviction = scanner_report[scanner_report['Energy_Gap'] > 1.0]
if not high_conviction.empty:
    print(f"\n🔥 HIGH CONVICTION ALERT: Focus on {list(high_conviction['Ticker'].values)}")
else:
    print("\n⚠️ No high-conviction signals detected. System suggests staying in cash or re-sampling.")

/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)
/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)
/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)
/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)
/tmp/ipykernel_40346/2792960372.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="1mo", interval="1d", progress=False)


--- 🎯 STRATEGIC SCANNER REPORT ---


,Ticker,Confidence_Prob,Energy_Gap,Status,Directive
0,PETR4.SA,0.964377,3.946579,HIGH,🚀 EXECUTE: Proceed with high-confidence implem...
1,VALE3.SA,0.955784,3.622452,HIGH,🚀 EXECUTE: Proceed with high-confidence implem...
2,ITUB4.SA,0.688268,0.821144,HIGH,🚀 EXECUTE: Proceed with high-confidence implem...
4,BBDC4.SA,0.656331,0.690437,HIGH,🚀 EXECUTE: Proceed with high-confidence implem...
3,ABEV3.SA,0.510545,0.077538,LOW,⚖️ REVIEW: Human-in-the-loop validation recomm...



🔥 HIGH CONVICTION ALERT: Focus on ['PETR4.SA', 'VALE3.SA']


### ⚖️ Quantitative Allocation & Risk Weighting
By combining the **Probability** and the **Energy Gap**, we create a single 'Conviction Score' that guides capital allocation. This prevents over-exposure to assets with high probability but low structural separation.

In [29]:
def calculate_allocation(df):
    # 1. Combined Conviction Score
    df['Conviction_Score'] = df['Confidence_Prob'] * df['Energy_Gap']

    # 2. Allocation Logic based on Tiers
    def assign_weight(row):
        if row['Confidence_Prob'] > 0.9 and row['Energy_Gap'] > 2.0:
            return 0.40  # Aggressive
        elif row['Confidence_Prob'] > 0.7 and row['Energy_Gap'] > 0.5:
            return 0.25  # Moderate
        elif row['Confidence_Prob'] > 0.6:
            return 0.15  # Defensive
        else:
            return 0.00  # Cash/Ignore

    df['Suggested_Weight'] = df.apply(assign_weight, axis=1)

    # Normalize weights to ensure they don't exceed 100% (optional, if using as portfolio)
    total_weight = df['Suggested_Weight'].sum()
    if total_weight > 1.0:
        df['Normalized_Weight'] = df['Suggested_Weight'] / total_weight
    else:
        df['Normalized_Weight'] = df['Suggested_Weight']

    return df.sort_values(by='Conviction_Score', ascending=False)

# Process the latest report
final_allocation_df = calculate_allocation(scanner_report)

print("--- 🏛️ PORTFOLIO ALLOCATION STRATEGY ---")
display(final_allocation_df[['Ticker', 'Conviction_Score', 'Suggested_Weight', 'Normalized_Weight', 'Directive']])

total_exposure = final_allocation_df['Suggested_Weight'].sum() * 100
print(f"\n📈 TOTAL PORTFOLIO EXPOSURE: {total_exposure:.1f}%")
print(f"💰 RESERVED CASH: {100 - total_exposure:.1f}%")

--- 🏛️ PORTFOLIO ALLOCATION STRATEGY ---


,Ticker,Conviction_Score,Suggested_Weight,Normalized_Weight,Directive
0,PETR4.SA,3.805991,0.40,0.363636,🚀 EXECUTE: Proceed with high-confidence implem...
1,VALE3.SA,3.462282,0.40,0.363636,🚀 EXECUTE: Proceed with high-confidence implem...
2,ITUB4.SA,0.565167,0.15,0.136364,🚀 EXECUTE: Proceed with high-confidence implem...
4,BBDC4.SA,0.453156,0.15,0.136364,🚀 EXECUTE: Proceed with high-confidence implem...
3,ABEV3.SA,0.039586,0.00,0.000000,⚖️ REVIEW: Human-in-the-loop validation recomm...



📈 TOTAL PORTFOLIO EXPOSURE: 110.0%
💰 RESERVED CASH: -10.0%


### 🛡️ Professional Portfolio Engine: Soft-Allocation & Risk Constraints
We are moving from 'Fixed Tiers' to 'Soft Allocation'. This approach uses a Softmax-like distribution to ensure capital is allocated proportionally to the model's conviction, while enforcing a strict 100% exposure limit and a mandatory cash reserve.

In [30]:
def professional_allocation(df, max_exposure=0.90, temperature=1.5):
    """
    Applies Soft-Max allocation based on Conviction Scores.
    - max_exposure: Max % of capital to deploy (e.g., 0.90 = 10% cash reserve)
    - temperature: Controls sensitivity. High = more uniform, Low = more concentrated on top signals.
    """
    scores = df['Conviction_Score'].values

    # 1. Softmax-based weights (Soft Allocation)
    exp_scores = np.exp(scores / temperature)
    softmax_weights = exp_scores / exp_scores.sum()

    # 2. Scale weights by Max Exposure constraint
    df['Final_Proportional_Weight'] = softmax_weights * max_exposure

    # 3. Calculate expected contribution to portfolio risk (Score * Weight)
    df['Strategic_Impact'] = df['Conviction_Score'] * df['Final_Proportional_Weight']

    return df.sort_values(by='Final_Proportional_Weight', ascending=False)

# Execute refined allocation
refined_portfolio = professional_allocation(final_allocation_df, max_exposure=0.85)

print("--- 🏛️ REFINED SOFT-ALLOCATION REPORT ---")
display(refined_portfolio[['Ticker', 'Conviction_Score', 'Final_Proportional_Weight', 'Strategic_Impact', 'Directive']])

actual_exposure = refined_portfolio['Final_Proportional_Weight'].sum() * 100
print(f"\n✅ TOTAL PORTFOLIO EXPOSURE: {actual_exposure:.1f}%")
print(f"🛡️ MANDATORY CASH RESERVE: {100 - actual_exposure:.1f}%")
print(f"🔥 TOP CONVICTION: {refined_portfolio.iloc[0]['Ticker']} gets {(refined_portfolio.iloc[0]['Final_Proportional_Weight']*100):.1f}% of total capital.")

--- 🏛️ REFINED SOFT-ALLOCATION REPORT ---


,Ticker,Conviction_Score,Final_Proportional_Weight,Strategic_Impact,Directive
0,PETR4.SA,3.805991,0.405024,1.541519,🚀 EXECUTE: Proceed with high-confidence implem...
1,VALE3.SA,3.462282,0.322082,1.115141,🚀 EXECUTE: Proceed with high-confidence implem...
2,ITUB4.SA,0.565167,0.046684,0.026384,🚀 EXECUTE: Proceed with high-confidence implem...
4,BBDC4.SA,0.453156,0.043325,0.019633,🚀 EXECUTE: Proceed with high-confidence implem...
3,ABEV3.SA,0.039586,0.032885,0.001302,⚖️ REVIEW: Human-in-the-loop validation recomm...



✅ TOTAL PORTFOLIO EXPOSURE: 85.0%
🛡️ MANDATORY CASH RESERVE: 15.0%
🔥 TOP CONVICTION: PETR4.SA gets 40.5% of total capital.


### 🚀 Re-Evaluating with a Trained Brain
Now that the model has seen real market patterns, let's re-run the analysis. The 'Energy Gap' should now be much more meaningful.